# Function Calling

**Learning goal:** Instead of asking Claude to respond in plain text, define a *tool* that forces Claude to return structured JSON. This is the mechanism that makes programmatic extraction reliable.

**Concepts covered**
- Defining a tool with a JSON schema
- Passing `tools` to `client.messages.create()`
- `tool_use` stop reason vs `end_turn`
- Extracting the tool call arguments from the response
- What happens when Claude cannot fill a required field

The `tools` parameter tells Claude what tools (or schemas) are available.

```python
response = client.messages.create(
    ...
    tools=[INTAKE_TOOL]
)
```

In this example:

```python
tools=[INTAKE_TOOL]
```

means:

> Claude, you may use the `extract_intake_fields` tool, and here is the JSON structure it should follow.

`INTAKE_TOOL` does **not** generate the JSON.

Instead:

* `INTAKE_TOOL` defines the JSON structure (schema).
* Claude reads the user's message.
* Claude fills the schema and generates the JSON output.

Think of it as:

```text
INTAKE_TOOL = Blank Form

Claude = Person filling the form

User Message = Information source
```

Flow:

```text
User Message
      │
      ▼
Claude reads message
      │
      ▼
Uses INTAKE_TOOL schema
      │
      ▼
Generates JSON
```

In this example, no actual Python function is executed. The tool is being used only as a schema to guide Claude's JSON output.

```python
print(f"Stop reason: {response.stop_reason}")
```

Displays why Claude stopped generating.

Typical values:

| Value | Meaning |
|---------|----------|
| `end_turn` | Claude answered normally |
| `tool_use` | Claude decided to call a tool |
| `max_tokens` | Token limit reached |

## Cell 1 — Setup

In [1]:
from dotenv import load_dotenv
import os
import anthropic
import json

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
MODEL = "claude-sonnet-4-6"

## Cell 2 — Define the Tool Schema

In [2]:
INTAKE_TOOL = {
    "name": "extract_intake_fields",
    "description": "Extract structured regulatory intake fields from a pharma compliance query.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query_type": {
                "type": "string",
                "enum": ["complaint", "submission", "variation", "safety_signal",
                         "label_update", "inspection", "general_enquiry"],
                "description": "The nature of the regulatory interaction."
            },
            "regulation_ref": {
                "type": "string",
                "enum": ["FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
                         "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"],
                "description": "The primary regulatory framework the query relates to."
            },
            "product_area": {
                "type": "string",
                "enum": ["oncology", "cardiovascular", "infectious_disease",
                         "cmc", "clinical", "labelling", "general"],
                "description": "The therapeutic area or functional domain."
            },
            "urgency": {
                "type": "string",
                "enum": ["routine", "standard", "urgent", "critical"],
                "description": "Deadline-driven priority level."
            },
            "submitting_team": {
                "type": "string",
                "description": "The team or function submitting the query. Never an individual name."
            }
        },
        "required": ["query_type", "regulation_ref", "product_area", "urgency", "submitting_team"]
    }
}

## Cell 3 — Call Claude with the Tool

In [3]:
SYSTEM_PROMPT = """
You are SmartIntake, a regulatory affairs intake specialist.
Use the extract_intake_fields tool to extract structured fields from the query.
Never infer urgency from tone alone — if deadline is unclear, set urgency to 'routine'
and flag it. submitting_team must be a team name, never an individual's name.
Think step by step before choosing enum values.
"""

def extract_fields(user_message):
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=SYSTEM_PROMPT,
        tools=[INTAKE_TOOL],
        tool_choice={"type": "auto"},
        messages=[{"role": "user", "content": user_message}]
    )

    print(f"Stop reason: {response.stop_reason}")

    if response.stop_reason == "tool_use":
        for block in response.content:
            if block.type == "tool_use":
                print(f"\nTool called: {block.name}")
                print(f"Arguments:\n{json.dumps(block.input, indent=2)}")
                return block.input
    else:
        print("Claude did not call the tool.")
        print(response.content[0].text)
        return None

result = extract_fields(
    "PV team here. We have a serious unexpected SUSAR for study NCT-2244 "
    "and need to report to EMA within 15 days per ICH E2A."
)

Stop reason: tool_use

Tool called: extract_intake_fields
Arguments:
{
  "query_type": "safety_signal",
  "regulation_ref": "ICH_E2A",
  "product_area": "clinical",
  "urgency": "critical",
  "submitting_team": "Pharmacovigilance Team"
}


## Cell 4 — Test with Partial Information

In your code:

```python
tool_choice={"type": "auto"}
```

`auto` means:

> Claude decides whether to use the tool or respond with normal text.

Claude checks whether it has enough information to fill all the required fields defined in the tool schema.

Example:

```python
"required": [
    "query_type",
    "regulation_ref",
    "product_area",
    "urgency",
    "submitting_team"
]
```

For this input:

```text
Hi, we need to file a Type II variation for our cardiovascular product with the EMA.
```

Claude can determine:

```text
query_type = variation
regulation_ref = EMA_CTR
product_area = cardiovascular
urgency = routine
```

But it cannot determine:

```text
submitting_team
```

Since a required field is missing, Claude chooses not to call the tool and instead asks a clarification question.

Flow:

```text
User Message
      │
      ▼
Does Claude have enough information?
      │
 ┌────┴────┐
 │         │
Yes        No
 │         │
 ▼         ▼
Call Tool  Ask Question
```

This is why you see:

```text
Stop reason: end_turn
```

instead of:

```text
Stop reason: tool_use
```

With `tool_choice="auto"`, Claude decides whether a tool call is appropriate based on the information available.

In [4]:
# What does Claude return when urgency is ambiguous?
result = extract_fields(
    "Hi, we need to file a Type II variation for our cardiovascular product with the EMA."
)

Stop reason: end_turn
Claude did not call the tool.
Let me analyze your query step by step before extracting the fields:

- **query_type**: "Type II variation" → `variation`
- **regulation_ref**: "EMA" → `EMA_CTR`
- **product_area**: "cardiovascular" → `cardiovascular`
- **urgency**: No deadline mentioned → `routine` (will flag)
- **submitting_team**: Not specified → I need to ask... but wait, let me check if it can be inferred. It cannot — I'll need to ask.

I have everything except the **submitting_team**. Could you please provide the name of the **team or function** submitting this query? (e.g., "Regulatory Affairs", "CMC Team", "Medical Affairs", etc.)

> ⚠️ **Urgency note**: No deadline was mentioned, so urgency will default to **routine**. Please let me know if there is a specific deadline so I can adjust the priority accordingly.


## Cell 5 — Test with Non-Regulatory Input

In [5]:
# What does Claude do with a non-regulatory query?
result = extract_fields("Can you check the weather in Mumbai today?")

Stop reason: end_turn
Claude did not call the tool.
I'm sorry, but checking the weather isn't something I'm able to help with. I'm **SmartIntake**, a regulatory affairs intake specialist designed to assist with **pharmaceutical regulatory compliance queries**.

Here's what I *can* help you with:
- 📋 **Submissions** – NDA, BLA, MAA, or other regulatory submissions
- ⚠️ **Safety Signals** – Adverse event reporting and pharmacovigilance
- 🏷️ **Label Updates** – Proposed changes to product labelling
- 🔍 **Inspections** – GMP/GCP inspection-related queries
- 📝 **Variations** – Post-approval changes
- 💬 **General Regulatory Enquiries** – Framework and compliance questions

If you have a **regulatory affairs query**, please share the details and I'll get it structured and logged right away!


### Key Takeaway

Function calling forces Claude to return structured data rather than free text. The `tool_use` stop reason tells you a tool was called; `end_turn` means Claude responded in text instead (useful for detecting non-regulatory inputs or when Claude needs to ask a question).